In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("GDELT-Notebook")
    .master("spark://spark-master:7077")
    .config("spark.jars", ",".join([
        "/opt/iceberg-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
        "/opt/iceberg-jars/awssdk-bundle-2.20.160.jar",
        "/opt/iceberg-jars/hadoop-aws-3.3.4.jar",
        "/opt/iceberg-jars/aws-java-sdk-bundle-1.12.262.jar",
    ]))
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "rest")
    .config("spark.sql.catalog.local.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.local.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.local.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.local.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.local.s3.path-style-access", "true")
    .config("spark.sql.catalog.local.s3.access-key-id", "admin")
    .config("spark.sql.catalog.local.s3.secret-access-key", "changeme123")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "changeme123")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")          # new
    .config("spark.hadoop.fs.s3a.path.style.access", "true")              # new
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")  # new
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")        # new
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

spark

In [3]:
print(spark.conf.get("spark.sql.catalog.local"))
# org.apache.iceberg.spark.SparkCatalog

org.apache.iceberg.spark.SparkCatalog


In [4]:
df = spark.sql("select * from local.bronze.events limit 10")
df.printSchema()

root
 |-- GLOBALEVENTID: long (nullable = true)
 |-- SQLDATE: string (nullable = true)
 |-- MonthYear: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- FractionDate: double (nullable = true)
 |-- Actor1Code: string (nullable = true)
 |-- Actor1Name: string (nullable = true)
 |-- Actor1CountryCode: string (nullable = true)
 |-- Actor1KnownGroupCode: string (nullable = true)
 |-- Actor1EthnicCode: string (nullable = true)
 |-- Actor1Religion1Code: string (nullable = true)
 |-- Actor1Religion2Code: string (nullable = true)
 |-- Actor1Type1Code: string (nullable = true)
 |-- Actor1Type2Code: string (nullable = true)
 |-- Actor1Type3Code: string (nullable = true)
 |-- Actor2Code: string (nullable = true)
 |-- Actor2Name: string (nullable = true)
 |-- Actor2CountryCode: string (nullable = true)
 |-- Actor2KnownGroupCode: string (nullable = true)
 |-- Actor2EthnicCode: string (nullable = true)
 |-- Actor2Religion1Code: string (nullable = true)
 |-- Actor2Religion2Code: stri

In [2]:
spark.sql("""
         select sqldate, count(*) from local.bronze.events group by sqldate order by sqldate desc 
    """).show(truncate=False)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------+--------+
|sqldate |count(1)|
+--------+--------+
|20260915|2639    |
|20260912|694     |
|20260908|50      |
|20260905|1       |
|20260816|10      |
|20260813|4       |
|20250915|9       |
|20250912|3       |
+--------+--------+



In [2]:
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jvm.java.net.URI("s3a://gdelt-landing"), hadoop_conf
)

def list_ts(stream):
    path = spark._jvm.org.apache.hadoop.fs.Path(f"s3a://gdelt-landing/{stream}/")
    return {str(s.getPath().getName()).replace(".csv", "") for s in fs.listStatus(path)}

complete = sorted(list_ts("events") & list_ts("mentions") & list_ts("gkg"))
print(f"{len(complete)} complete batches; most recent:", complete[-1] if complete else None)

26/09/21 12:43:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


1 complete batches; most recent: 20260912183000


In [3]:
chosen_ts = complete[-1]
spark.read.option("sep", "\t").option("inferSchema", "true") \
    .csv(f"s3a://gdelt-landing/events/{chosen_ts}.csv") \
    .show(10, truncate=False)

26/09/21 12:43:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+--------+------+----+---------+------+----------+----+----+----+----+----+----+----+----+------+----------+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+----+----+----+-----------------+----+----------------------------------+----+----+-----+-------+-------+--------+----+-----------------------------------+----+----+-----+-------+-------+--------+----+-----------------------------------+----+----+-----+-------+-------+--------+--------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|_c0       |_c1     |_c2   |_c3 |_c4      |_c5   |_c6       |_c7 |_c8 |_c9 |_c10|_c11|_c12|_c13|_c14|_c15  |_c16      |_c17|_c18|_c19|_c20|_c21|_c22|_c23|_c24|_c25|_c26|_c27|_c28|_c29|_c30 |_c31|_c32|_c33|_c34             |_c35|_c36                              |_c37|_c38|_c39 |_c40   |_c41   |_c42    |_c43|_c44                               |_c45|_c46|_c47 |_c48   |_c49  

26/09/21 13:13:11 WARN StandaloneAppClient$ClientEndpoint: Connection to 3192b8f39afb:7077 failed; waiting for master to reconnect...
26/09/21 13:13:11 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/09/21 13:19:49 ERROR TaskSchedulerImpl: Lost executor 0 on 172.21.0.7: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.


In [6]:
spark.sql("""
    SELECT committed_at, operation,
       summary['added-data-files'] AS files_added,
       summary['added-records']    AS rows_added,
       summary['total-records']    AS total_rows_after
FROM local.bronze.events.snapshots
ORDER BY committed_at DESC;
          """).show(truncate=False)

+-----------------------+---------+-----------+----------+----------------+
|committed_at           |operation|files_added|rows_added|total_rows_after|
+-----------------------+---------+-----------+----------+----------------+
|2026-09-23 11:15:21.602|append   |1          |1631      |61197           |
|2026-09-23 10:55:27.023|append   |1          |1458      |59566           |
|2026-09-22 21:45:32.968|append   |1          |1094      |58108           |
|2026-09-22 21:30:17.523|append   |1          |1522      |57014           |
|2026-09-22 21:15:18.951|append   |1          |961       |55492           |
|2026-09-22 21:00:19.196|append   |1          |1569      |54531           |
|2026-09-22 20:45:18.901|append   |1          |1390      |52962           |
|2026-09-22 20:30:18.993|append   |1          |1156      |51572           |
|2026-09-22 20:15:30.207|append   |1          |1714      |50416           |
|2026-09-22 20:00:18.354|append   |1          |1336      |48702           |
|2026-09-22 

In [7]:
spark.sql("""
         SELECT s.committed_at,
       s.operation,
       e.data_file.file_path,
       e.data_file.record_count,
       e.data_file.file_size_in_bytes,
       e.data_file.partition
FROM local.bronze.events.entries e
JOIN local.bronze.events.snapshots s
  ON e.snapshot_id = s.snapshot_id
WHERE e.status = 1   -- ADDED (0=EXISTING, 2=DELETED — check with a SELECT DISTINCT status if unsure)
ORDER BY s.committed_at DESC, e.data_file.file_path; 
    """).show(truncate=False)

+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------+------------+------------------+------------+
|committed_at           |operation|file_path                                                                                                                   |record_count|file_size_in_bytes|partition   |
+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------+------------+------------------+------------+
|2026-09-23 11:15:21.602|append   |s3://warehouse/bronze/events/data/date_added_ts_day=2026-09-23/00000-29-b7b1f535-88dd-47c4-a223-ad2b710882a0-0-00001.parquet|1631        |120119            |{2026-09-23}|
|2026-09-23 10:55:27.023|append   |s3://warehouse/bronze/events/data/date_added_ts_day=2026-09-23/00000-29-00f69dd0-43c4-46f2-8676-5163e36dc345-0-00001.parquet|1458        |112

In [9]:
spark.sql("""
    SELECT partition, record_count, file_count, total_data_file_size_in_bytes
FROM local.bronze.events.partitions
ORDER BY partition DESC;
          """).show(truncate=False)

+------------+------------+----------+-----------------------------+
|partition   |record_count|file_count|total_data_file_size_in_bytes|
+------------+------------+----------+-----------------------------+
|{2026-09-23}|3089        |2         |232284                       |
|{2026-09-22}|53865       |36        |4125304                      |
|{2026-09-20}|833         |1         |75237                        |
|{2026-09-15}|2708        |3         |244027                       |
|{2026-09-12}|702         |1         |70510                        |
+------------+------------+----------+-----------------------------+



In [10]:
spark.sql(""" 
         SELECT 'events' AS stream, MAX(committed_at) AS last_ingest FROM local.bronze.events.snapshots
UNION ALL
SELECT 'mentions', MAX(committed_at) FROM local.bronze.mentions.snapshots
UNION ALL
SELECT 'gkg', MAX(committed_at) FROM local.bronze.gkg.snapshots; 
          """).show(truncate=False)

+--------+-----------------------+
|stream  |last_ingest            |
+--------+-----------------------+
|events  |2026-09-23 11:15:21.602|
|mentions|2026-09-23 11:15:31.466|
|gkg     |2026-09-23 10:55:06.378|
+--------+-----------------------+



In [11]:
spark.sql(""" 
          SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GLOBALEVENTID) AS distinct_keys,
       SUM(CASE WHEN GLOBALEVENTID IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(date_added_ts) AS earliest_ts,
       MAX(date_added_ts) AS latest_ts
FROM local.bronze.events;
          """).show(truncate=False)

+----------+-------------+---------+-------------------+-------------------+
|total_rows|distinct_keys|null_keys|earliest_ts        |latest_ts          |
+----------+-------------+---------+-------------------+-------------------+
|61197     |61197        |0        |2026-09-12 18:30:00|2026-09-23 11:15:00|
+----------+-------------+---------+-------------------+-------------------+



In [12]:
spark.sql(""" 
         SELECT date_trunc('day', date_added_ts) AS day, COUNT(*) AS row_count
FROM local.bronze.events
GROUP BY 1
ORDER BY 1 DESC; 
          """).show(truncate=False)

+-------------------+---------+
|day                |row_count|
+-------------------+---------+
|2026-09-23 00:00:00|3089     |
|2026-09-22 00:00:00|53865    |
|2026-09-20 00:00:00|833      |
|2026-09-15 00:00:00|2708     |
|2026-09-12 00:00:00|702      |
+-------------------+---------+



In [13]:
spark.sql(""" 
         -- bronze.events
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GLOBALEVENTID) AS distinct_keys,
       SUM(CASE WHEN GLOBALEVENTID IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(date_added_ts) AS earliest_ts,
       MAX(date_added_ts) AS latest_ts
FROM local.bronze.events; 
          """).show(truncate=False)

+----------+-------------+---------+-------------------+-------------------+
|total_rows|distinct_keys|null_keys|earliest_ts        |latest_ts          |
+----------+-------------+---------+-------------------+-------------------+
|61197     |61197        |0        |2026-09-12 18:30:00|2026-09-23 11:15:00|
+----------+-------------+---------+-------------------+-------------------+



In [14]:
spark.sql(""" 
         -- bronze.mentions (composite key includes mention_ts, per dedup_and_append)
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GLOBALEVENTID, MentionIdentifier, mention_ts) AS distinct_keys,
       SUM(CASE WHEN GLOBALEVENTID IS NULL OR MentionIdentifier IS NULL OR mention_ts IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(mention_ts) AS earliest_ts,
       MAX(mention_ts) AS latest_ts
FROM local.bronze.mentions; 
          """).show(truncate=False)

+----------+-------------+---------+-------------------+-------------------+
|total_rows|distinct_keys|null_keys|earliest_ts        |latest_ts          |
+----------+-------------+---------+-------------------+-------------------+
|192390    |187615       |0        |2026-09-12 18:30:00|2026-09-23 11:15:00|
+----------+-------------+---------+-------------------+-------------------+



In [15]:
spark.sql(""" 
         -- bronze.gkg
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GKGRECORDID) AS distinct_keys,
       SUM(CASE WHEN GKGRECORDID IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(gkg_ts) AS earliest_ts,
       MAX(gkg_ts) AS latest_ts
FROM local.bronze.gkg; 
          """).show(truncate=False)

+----------+-------------+---------+-------------------+-------------------+
|total_rows|distinct_keys|null_keys|earliest_ts        |latest_ts          |
+----------+-------------+---------+-------------------+-------------------+
|56089     |56089        |0        |2026-01-01 00:15:00|2026-09-22 21:15:00|
+----------+-------------+---------+-------------------+-------------------+



In [16]:
spark.sql(""" 
         -- silver.events_india
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GLOBALEVENTID) AS distinct_keys,
       SUM(CASE WHEN GLOBALEVENTID IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(event_date) AS earliest_date,
       MAX(event_date) AS latest_date
FROM local.silver.events_india; 
          """).show(truncate=False)

+----------+-------------+---------+-------------+-----------+
|total_rows|distinct_keys|null_keys|earliest_date|latest_date|
+----------+-------------+---------+-------------+-----------+
|2974      |2974         |0        |2016-09-24   |2026-09-23 |
+----------+-------------+---------+-------------+-----------+



In [17]:
spark.sql(""" 
         -- silver.mentions_enriched (no mention_ts retained here, so key is GLOBALEVENTID+MentionIdentifier —
-- if bronze had two mentions of the same event by the same source at different mention_ts, this key will legitimately collapse them; check that's expected)
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GLOBALEVENTID, MentionIdentifier) AS distinct_keys,
       SUM(CASE WHEN GLOBALEVENTID IS NULL OR MentionIdentifier IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(event_date) AS earliest_date,
       MAX(event_date) AS latest_date
FROM local.silver.mentions_enriched; 
          """).show(truncate=False)

+----------+-------------+---------+-------------+-----------+
|total_rows|distinct_keys|null_keys|earliest_date|latest_date|
+----------+-------------+---------+-------------+-----------+
|3940      |3905         |0        |2016-09-24   |2026-09-23 |
+----------+-------------+---------+-------------+-----------+



In [18]:
spark.sql(""" 
          -- gold.india_bilateral_sentiment_daily (grouped by event_date, partner_country)
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_date, partner_country) AS distinct_keys,
       SUM(CASE WHEN event_date IS NULL OR partner_country IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(event_date) AS earliest_date,
       MAX(event_date) AS latest_date
FROM local.gold.india_bilateral_sentiment_daily;
          """).show(truncate=False)

+----------+-------------+---------+-------------+-----------+
|total_rows|distinct_keys|null_keys|earliest_date|latest_date|
+----------+-------------+---------+-------------+-----------+
|90        |90           |0        |2025-09-22   |2026-09-23 |
+----------+-------------+---------+-------------+-----------+



In [19]:
spark.sql(""" 
          -- gold.india_state_conflict_daily (grouped by event_date, adm1_code, conflict_severity)
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_date, adm1_code, conflict_severity) AS distinct_keys,
       SUM(CASE WHEN event_date IS NULL OR adm1_code IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(event_date) AS earliest_date,
       MAX(event_date) AS latest_date
FROM local.gold.india_state_conflict_daily;
          """).show(truncate=False)

+----------+-------------+---------+-------------+-----------+
|total_rows|distinct_keys|null_keys|earliest_date|latest_date|
+----------+-------------+---------+-------------+-----------+
|317       |317          |0        |2016-09-24   |2026-09-23 |
+----------+-------------+---------+-------------+-----------+



In [20]:
spark.sql(""" 
          -- gold.india_event_mention_counts (MERGE-upserted, so GLOBALEVENTID should always be unique)
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT GLOBALEVENTID) AS distinct_keys,
       SUM(CASE WHEN GLOBALEVENTID IS NULL THEN 1 ELSE 0 END) AS null_keys,
       MIN(last_updated_ts) AS earliest_ts,
       MAX(last_updated_ts) AS latest_ts
FROM local.gold.india_event_mention_counts;
          """).show(truncate=False)

+----------+-------------+---------+--------------------------+--------------------------+
|total_rows|distinct_keys|null_keys|earliest_ts               |latest_ts                 |
+----------+-------------+---------+--------------------------+--------------------------+
|2911      |2911         |0        |2026-09-23 10:56:12.700838|2026-09-23 10:56:12.700838|
+----------+-------------+---------+--------------------------+--------------------------+



In [21]:
spark.sql(""" 
          
          SELECT GLOBALEVENTID, GoldsteinScale, AvgTone, QuadClass, ActionGeo_Lat, ActionGeo_Long
FROM local.bronze.events
WHERE GoldsteinScale NOT BETWEEN -10 AND 10
   OR QuadClass NOT IN (1,2,3,4)
   OR ABS(AvgTone) > 100
   OR ActionGeo_Lat NOT BETWEEN -90 AND 90
   OR ActionGeo_Long NOT BETWEEN -180 AND 180;
 """).show(truncate=False)

+-------------+--------------+-------+---------+-------------+--------------+
|GLOBALEVENTID|GoldsteinScale|AvgTone|QuadClass|ActionGeo_Lat|ActionGeo_Long|
+-------------+--------------+-------+---------+-------------+--------------+
+-------------+--------------+-------+---------+-------------+--------------+



In [22]:
spark.sql(""" 
          SELECT GLOBALEVENTID, QuadClass, conflict_severity
FROM local.silver.events_india
WHERE conflict_severity = 'Verbal Cooperation'
  AND (QuadClass IS NULL OR QuadClass NOT IN (1,2,3,4));
          
 """).show(truncate=False)

+-------------+---------+-----------------+
|GLOBALEVENTID|QuadClass|conflict_severity|
+-------------+---------+-----------------+
+-------------+---------+-----------------+



In [23]:
spark.sql(""" 
        SELECT norm_url, COUNT(*) AS c
FROM (
  SELECT LOWER(TRIM(REGEXP_REPLACE(DocumentIdentifier, '\\?.*$', ''))) AS norm_url
  FROM local.bronze.gkg
)
GROUP BY norm_url
HAVING COUNT(*) > 1
ORDER BY c DESC;  
          
 """).show(truncate=False)

26/09/23 11:39:10 WARN TaskSetManager: Lost task 0.0 in stage 74.0 (TID 130) (172.21.0.7 executor 0): org.apache.spark.SparkRuntimeException: [INVALID_PARAMETER_VALUE.PATTERN] The value of parameter(s) `regexp` in `regexp_replace` is invalid: '?.*$'.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidPatternError(QueryExecutionErrors.scala:2620)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidPatternError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:43)
	at scala.collectio

Py4JJavaError: An error occurred while calling o140.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 74.0 failed 4 times, most recent failure: Lost task 0.3 in stage 74.0 (TID 136) (172.21.0.7 executor 0): org.apache.spark.SparkRuntimeException: [INVALID_PARAMETER_VALUE.PATTERN] The value of parameter(s) `regexp` in `regexp_replace` is invalid: '?.*$'.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidPatternError(QueryExecutionErrors.scala:2620)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidPatternError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.executi

In [24]:
spark.sql(""" 
         SELECT s.GLOBALEVENTID
FROM local.silver.events_india s
LEFT ANTI JOIN local.bronze.events b
  ON s.GLOBALEVENTID = b.GLOBALEVENTID; 
          
 """).show(truncate=False)

+-------------+
|GLOBALEVENTID|
+-------------+
+-------------+



In [25]:
spark.sql(""" 
       SELECT g.GLOBALEVENTID, g.total_mentions, g.last_updated_ts
FROM local.gold.india_event_mention_counts g
LEFT ANTI JOIN local.silver.mentions_enriched s
  ON g.GLOBALEVENTID = s.GLOBALEVENTID;   
          
 """).show(truncate=False)

+-------------+--------------+---------------+
|GLOBALEVENTID|total_mentions|last_updated_ts|
+-------------+--------------+---------------+
+-------------+--------------+---------------+



In [26]:
spark.sql(""" 
          SELECT date_trunc('day', date_added_ts) AS day,
       COUNT(*) AS bronze_rows,
       SUM(CASE WHEN ActionGeo_CountryCode = 'IN'
                  OR Actor1CountryCode = 'IND'
                  OR Actor2CountryCode = 'IND' THEN 1 ELSE 0 END) AS india_matching_rows
FROM local.bronze.events
GROUP BY 1
ORDER BY 1 DESC;
    """).show(truncate=False)

+-------------------+-----------+-------------------+
|day                |bronze_rows|india_matching_rows|
+-------------------+-----------+-------------------+
|2026-09-23 00:00:00|4244       |169                |
|2026-09-22 00:00:00|53865      |2407               |
|2026-09-20 00:00:00|833        |72                 |
|2026-09-15 00:00:00|2708       |298                |
|2026-09-12 00:00:00|702        |78                 |
+-------------------+-----------+-------------------+



In [27]:
spark.sql(""" 
          SELECT m.GLOBALEVENTID, m.mention_ts, e.date_added_ts
FROM local.bronze.mentions m
JOIN local.bronze.events e
  ON m.GLOBALEVENTID = e.GLOBALEVENTID
WHERE m.mention_ts < e.date_added_ts
LIMIT 50;
    """).show(truncate=False)

+-------------+----------+-------------+
|GLOBALEVENTID|mention_ts|date_added_ts|
+-------------+----------+-------------+
+-------------+----------+-------------+



In [28]:
spark.sql(""" 
          
    """).show(truncate=False)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near end of input.(line 3, pos 4)

== SQL ==
 

    
----^^^


In [ ]:
spark.sql(""" 
          
    """).show(truncate=False)

In [ ]:
spark.sql(""" 
          
    """).show(truncate=False)

In [ ]:
spark.sql(""" 
          
    """).show(truncate=False)

In [ ]:
spark.sql(""" 
          
    """).show(truncate=False)